# ⚛️ quantum-automl — Quickstart Notebook

> **"You bring the data, we build the quantum model."**

This notebook demonstrates the three main use-cases:
1. Binary classification on synthetic data
2. Multi-class classification on the Iris dataset
3. Regression on the Diabetes dataset

All examples are configured for **low-end hardware** (laptop / Colab free tier).


In [ ]:
# ── Install (run once) ────────────────────────────────────────────────────
# !pip install quantum-automl
# For visualisations and XAI:
# !pip install "quantum-automl[advanced]"

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    r2_score, mean_squared_error
)

from quantum_automl import QuantumAutoClassifier, QuantumAutoRegressor
from quantum_automl.utils import setup_logging, check_hardware_compatibility

setup_logging('WARNING')   # set to 'DEBUG' for maximum verbosity

# Quick hardware check
print('Hardware compatibility check:')
for n in [2, 4, 6, 8]:
    check_hardware_compatibility(n)

---
## 1️⃣  Binary Classification — Synthetic Data

We generate a 4-feature, 2-class dataset and let `QuantumAutoClassifier` find the best quantum model automatically.

⚡ **Low-end tip:** `max_qubits=4`, `max_iter=50`, `subsample=80` keeps runtime under ~2 min on a modern laptop.

In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=120,
    n_features=4,
    n_informative=3,
    n_redundant=1,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'Class balance: {np.bincount(y_train)}')

In [ ]:
clf_binary = QuantumAutoClassifier(
    max_qubits=4,
    max_iter=50,
    reps=1,
    cv_folds=3,
    subsample=80,
    include_kernel_models=True,
    early_stop_threshold=0.90,
    verbose=True,
)

clf_binary.fit(X_train, y_train)

In [ ]:
y_pred = clf_binary.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f'\n🎯 Test Accuracy : {acc:.4f}')
print(f'Best model      : {clf_binary.best_params_}')
print(f'Best CV score   : {clf_binary.best_score_:.4f}')
print()
print(classification_report(y_test, y_pred))

In [ ]:
# ── Confusion matrix ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Class 0', 'Class 1']).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix — Binary')

# Search score history
results = clf_binary.search_report_.all_results
names = [r.spec.name.replace('VQC_', '').replace('QSVC_', 'K:') for r in results if r.error is None]
scores = [r.cv_score_mean for r in results if r.error is None]
axes[1].barh(range(len(names)), scores, color='steelblue', alpha=0.8)
axes[1].set_yticks(range(len(names)))
axes[1].set_yticklabels(names, fontsize=7)
axes[1].set_xlabel('CV Accuracy')
axes[1].set_title('Model Search Results')
axes[1].axvline(clf_binary.best_score_, color='red', linestyle='--', label=f'Best={clf_binary.best_score_:.3f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('binary_classification_results.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 2️⃣  Multi-Class Classification — Iris Dataset

Iris has 4 features and 3 classes — a classic benchmark. `QuantumAutoClassifier` handles multi-class automatically via VQC's built-in one-vs-rest extension.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X_iris, y_iris = iris.data, iris.target

X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_iris, y_iris, test_size=0.25, random_state=42, stratify=y_iris
)

print(f'Iris — Train: {X_train_i.shape}  Test: {X_test_i.shape}')
print(f'Classes: {iris.target_names}')

In [ ]:
clf_iris = QuantumAutoClassifier(
    max_qubits=4,
    max_iter=50,
    reps=1,
    cv_folds=3,
    subsample=90,
    include_kernel_models=True,
    verbose=True,
)

clf_iris.fit(X_train_i, y_train_i)

In [ ]:
y_pred_i = clf_iris.predict(X_test_i)
acc_i = accuracy_score(y_test_i, y_pred_i)

print(f'\n🎯 Iris Test Accuracy : {acc_i:.4f}')
print(f'Best model           : {clf_iris.best_params_}')
print()
print(classification_report(y_test_i, y_pred_i, target_names=iris.target_names))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
cm_i = confusion_matrix(y_test_i, y_pred_i)
ConfusionMatrixDisplay(cm_i, display_labels=iris.target_names).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix — Iris (3-class)')

# Feature importance via data profile
feature_names = iris.feature_names
axes[1].bar(feature_names, np.std(X_iris, axis=0), color='coral', alpha=0.8)
axes[1].set_xlabel('Feature')
axes[1].set_ylabel('Std Dev (proxy for importance)')
axes[1].set_title('Feature Variance — Iris')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('iris_classification_results.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 3️⃣  Regression — Diabetes Dataset

The Diabetes dataset has 10 features and a continuous target.  We reduce to 4 features via PCA automatically.

In [ ]:
from sklearn.datasets import load_diabetes

diabetes = load_diabetes()
X_diab, y_diab = diabetes.data, diabetes.target

# Use a small subset for fast simulation
rng = np.random.default_rng(42)
idx = rng.choice(len(y_diab), 100, replace=False)
X_d, y_d = X_diab[idx], y_diab[idx]

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_d, y_d, test_size=0.25, random_state=42
)

print(f'Diabetes — Train: {X_train_d.shape}  Test: {X_test_d.shape}')
print(f'Target range: [{y_d.min():.0f}, {y_d.max():.0f}]')

In [ ]:
reg = QuantumAutoRegressor(
    max_qubits=4,
    max_iter=50,
    reps=1,
    cv_folds=3,
    subsample=60,
    include_kernel_models=True,
    verbose=True,
)

reg.fit(X_train_d, y_train_d)

In [ ]:
y_pred_d = reg.predict(X_test_d)
r2 = r2_score(y_test_d, y_pred_d)
rmse = np.sqrt(mean_squared_error(y_test_d, y_pred_d))

print(f'\n📈 Regression Results')
print(f'R²   : {r2:.4f}')
print(f'RMSE : {rmse:.2f}')
print(f'Best model : {reg.best_params_}')
print(f'Best CV R² : {reg.best_score_:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Predicted vs actual
axes[0].scatter(y_test_d, y_pred_d, alpha=0.7, color='steelblue', edgecolors='white')
lims = [min(y_test_d.min(), y_pred_d.min()), max(y_test_d.max(), y_pred_d.max())]
axes[0].plot(lims, lims, 'r--', lw=1.5, label='Perfect fit')
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
axes[0].set_title(f'Predicted vs Actual  (R²={r2:.3f})')
axes[0].legend()

# Residuals
residuals = y_test_d - y_pred_d
axes[1].hist(residuals, bins=12, color='coral', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='black', linestyle='--', lw=1)
axes[1].set_xlabel('Residual (Actual − Predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.savefig('regression_results.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 4️⃣  Bonus: Optuna Hyperparameter Search

Switch to Bayesian optimisation by setting `search_strategy='optuna'`.  Useful when you want smarter exploration with fewer total model evaluations.

In [ ]:
# Uncomment to run (requires optuna — pip install optuna)

# clf_optuna = QuantumAutoClassifier(
#     max_qubits=4,
#     max_iter=50,
#     search_strategy='optuna',
#     n_trials=15,
#     cv_folds=3,
#     subsample=80,
#     verbose=True,
# )
# clf_optuna.fit(X_train, y_train)
# print('Optuna best score:', clf_optuna.best_score_)
# print('Optuna best params:', clf_optuna.best_params_)

---
## 5️⃣  Bonus: XAI with QuantumExplainer

SHAP values explain which features most influenced each prediction.

In [ ]:
# Uncomment to run (requires shap — pip install shap)

# from quantum_automl.explainability import QuantumExplainer

# explainer = QuantumExplainer(clf_binary, background_samples=20)
# explainer.fit(X_train)
# shap_values = explainer.explain(X_test[:15], nsamples=50)

# importance = explainer.feature_importance(shap_values)
# print('Feature importance:', importance)

# explainer.summary_plot(shap_values, X_test[:15])

---
## 6️⃣  Bonus: Unsupervised Quantum Clustering

In [ ]:
# from quantum_automl.cluster import QuantumAutoCluster
# from sklearn.datasets import make_blobs

# X_blob, y_true = make_blobs(n_samples=60, n_features=4, centers=3, random_state=42)

# qcluster = QuantumAutoCluster(
#     n_clusters=3,
#     max_qubits=4,
#     subsample=60,
#     verbose=True,
# )
# labels = qcluster.fit_predict(X_blob)

# print('Cluster labels:', labels)
# print('Best feature map:', qcluster.best_feature_map_name_)
# print('Silhouette score:', qcluster.best_score_)

---
## 7️⃣  Circuit Visualisation

Inspect the best quantum circuit discovered by AutoML.

In [ ]:
from quantum_automl.utils import visualize_circuit

# Text output — always works, no matplotlib needed
print(visualize_circuit(clf_binary, style='text'))

In [ ]:
# Summary of all experiments
print('\n=== Experiment Summary ===')
print(f'Binary classification test accuracy : {acc:.4f}')
print(f'Iris  classification test accuracy  : {acc_i:.4f}')
print(f'Diabetes regression R²              : {r2:.4f}')